# Recruitment assay

**What it does.** Quantify how strongly a marker is recruited to the pathogen relative to the surrounding cytoplasm.

**When to use it.** For host-pathogen imaging where the readout is localisation rather than abundance.

**What you get.** Per-object recruitment ratios and per-condition summary plots.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.submodules.analyze_recruitment`

```
analyze_recruitment(settings)
```

Quantify recruitment of a fluorescent marker to the pathogenic vacuole and produce per-PV / per-well summaries.

In [ ]:
from spacr.submodules import analyze_recruitment

## 3. Settings

`spacr.settings.get_analyze_recruitment_default_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_analyze_recruitment_default_settings

defaults = get_analyze_recruitment_default_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (int) - Recruitment analysis only (analyze_recruitment): the
    # image-channel index paired with the cell mask when drawing outline
    # overlays, and the switch that enables the cell filters - set an
    # integer and cell_size_range, cell_intensity_range and
    # target_intensity_min are applied; leave it None and cells are not
    # filtered at all. Default 3.
    'cell_chann_dim': 3,

    # (list) - [min, max] bounds on a cell's mean intensity, applied
    # when the measurement table is filtered during recruitment analysis
    # (only when cell_chann_dim is set). Beware the channel it actually
    # tests: _object_filter is called with
    # mask_chans=[nucleus_chann_dim, pathogen_chann_dim, cell_chann_dim]
    # and index 0, so the column compared is
    # cell_channel_{nucleus_chann_dim}_mean_intensity - the cell object
    # measured in the NUCLEUS channel, not the cell channel. Bounds are
    # exclusive (a cell is kept when its value is > min and < max) and
    # both entries must be integers, since a float or None silently
    # skips that bound. Units are raw image intensity. Default [0,
    # 100000].
    'cell_intensity_range': [0, 100000],

    # (int) - Position along the last axis of each merged/*.npy array
    # where the cell label mask sits. Merged arrays are ordered [image
    # channels..., cell, nucleus, pathogen, organelle], so the default 4
    # assumes the four channels 0-3 were kept; keep fewer channels and
    # every mask dim shifts down. None makes measure_crop skip all cell
    # measurements and cell crops. Default 4.
    'cell_mask_dim': 4,

    # (list of lists) - Plate wells occupied by each entry of
    # cell_types, one inner list per cell type in the same order, e.g.
    # [['c2','c3'],['c4']]. Every identifier must start with 'c'
    # (column) or 'r' (row); anything else is silently skipped and those
    # wells get no host_cells label. An unlabelled well is not
    # necessarily lost: 'condition' is the join of whichever of host
    # cell / pathogen / treatment labels are present, so
    # analyze_recruitment only drops rows that have none of the three -
    # a well missing just the cell label lands in a different condition
    # group instead. plot_data_from_db is stricter and does drop rows
    # with no host_cells label whenever this key is set. Default None,
    # which labels every row with cell_types[0].
    'cell_plate_metadata': None,

    # (list) - [min, max] bounds in pixels^2 on cell_area, used to drop
    # rows from the measurement table during recruitment analysis; only
    # cells strictly between the two values are kept. Both entries must
    # be integers or that bound is silently skipped. Setting it to None
    # widens it to [0, 1e100]. Default [0, 100000].
    'cell_size_range': [0, 100000],

    # (list) - Names of the host cell lines in the experiment, e.g.
    # ['HeLa']. Each name is written into the host_cells column and
    # folded into the combined condition label used for grouping and
    # plotting; the list is positionally paired with
    # cell_plate_metadata, which says which wells hold each one. Default
    # ['HeLa'].
    'cell_types': ['HeLa'],

    # (int) - Minimum cells a well must contribute to survive
    # recruitment analysis; wells below it, and every cell in them, are
    # dropped before the by-well plots and CSVs are produced. Raise it
    # to suppress noisy, sparsely populated wells at the cost of losing
    # those wells. Default 0, which keeps every well.
    'cells_per_well': 0,

    # (list) - Recruitment analysis only: the image-channel indices held
    # in the merged arrays. They are handed to plot_image_mask_overlay
    # so the overlay/outline figures cover those channels, and the same
    # list drives the recruitment loop - but _calculate_recruitment
    # writes fixed, channel-less column names
    # ('pathogen_cell_mean_mean', 'pathogen_cytoplasm_q75_mean', ...),
    # so each pass overwrites the previous one: the column count never
    # changes and only the LAST index in the list determines which
    # channel's recruitment ratios survive. Put the channel you actually
    # want ratios for last, and trim the list only to cut plotting work.
    # Default [0,1,2,3].
    'channel_dims': [0, 1, 2, 3],

    # (int) - Index of the fluorescence channel the downstream analysis
    # focuses on. It decides which channel's features survive filtering
    # (other channels' features are dropped), defines recruitment =
    # pathogen_channel_N_mean_intensity /
    # cytoplasm_channel_N_mean_intensity, and is written into the ML
    # result paths. Set it to the channel carrying your phenotype
    # readout. Valid 0-3; default 3 in the ML/recruitment steps, 1-2
    # elsewhere.
    'channel_of_interest': 2,

    # (int) - Base figure size in inches; figures are built square as
    # figuresize x figuresize and font sizes are derived from it
    # (legend, axis labels and ticks at 0.75x, overlay text at 0.5x).
    # Raise it when text is unreadable at publication scale, lower it to
    # fit panels on screen. Default 10; cluster grids cap total width at
    # 200 inches.
    'figuresize': 10,

    # (int, bool, or None) - Cap on nuclei per cell applied when the
    # per-object tables are merged for analysis: None disables the
    # filter, True keeps only single-nucleus cells, an integer N keeps
    # cells with N or fewer nuclei. Cells over the cap are dropped
    # entirely from the merged table. Do not pass False - it is read as
    # 0 and removes everything. Defaults differ sharply by pipeline: 1
    # for plot-merge and recruitment, 2 for plot-data-from-db, True for
    # screen analysis and training-dataset generation, 10 for
    # endodyogeny, and 1000 (effectively off) for vision-model
    # interpretation and class-proportion analysis - check the pipeline
    # you are running rather than assuming.
    'nuclei_limit': 1,

    # (int) - Recruitment analysis only (analyze_recruitment): the
    # image-channel index paired with the nucleus mask when drawing
    # outline overlays, and the switch that enables nucleus_size_range /
    # nucleus_intensity_range filtering. Set it to None to skip nucleus
    # filtering. It plays no part in segmentation - use nucleus_channel
    # for that. Default 0.
    'nucleus_chann_dim': 0,

    # (list) - Two-element [min, max] bound on mean nucleus-channel
    # intensity used by the recruitment analysis to drop rows from the
    # measurement table - it filters measured objects, not masks or
    # normalization. Rows are kept only if min < mean intensity < max
    # (raw units), and each bound is ignored unless it is an int.
    # Default [0, 100000].
    'nucleus_intensity_range': [0, 100000],

    # (int) - Position along the last axis of each merged/*.npy array
    # where the nucleus label mask sits, one plane after the cell mask.
    # With the default four image channels (0-3) that is 5; keep a
    # different number of channels and it shifts by the same amount.
    # None makes measure_crop skip nucleus measurements and
    # cell-to-nucleus linking. Default 5.
    'nucleus_mask_dim': 5,

    # (list) - Two-element [min, max] bound in pixels^2 on nucleus_area,
    # used by the recruitment analysis to drop rows from the measurement
    # table; masks are left untouched. Rows are kept only if min < area
    # < max, and each bound is ignored unless it is an int. Default [0,
    # 100000]; None widens it to [0, 1e100].
    'nucleus_size_range': [0, 100000],

    # (int) - Recruitment analysis only (analyze_recruitment): the
    # image-channel index paired with the pathogen mask when drawing
    # outline overlays, and the switch that enables pathogen_size_range
    # / pathogen_intensity_range filtering. Set it to None to skip
    # pathogen filtering. It plays no part in segmentation - use
    # pathogen_channel for that. Default 2.
    'pathogen_chann_dim': 2,

    # (list) - Two-element [min, max] mean-intensity filter applied to
    # the pathogen table in analyze_recruitment; pathogens whose mean
    # intensity in the paired mask channel falls outside the open
    # interval are dropped before recruitment ratios are computed.
    # Bounds must be ints - floats are silently ignored. Default [0,
    # 100000]. Use it to exclude dead or saturated parasites.
    'pathogen_intensity_range': [0, 100000],

    # (int, bool, or None) - Maximum pathogens per cell. True or 1 =
    # single pathogen only; None or False = no limit; int = custom
    # limit.
    'pathogen_limit': 10,

    # (int) - Position along the last axis of each merged/*.npy array
    # where the pathogen label mask sits, one plane after the nucleus
    # mask. With the default four image channels (0-3) that is 6; shift
    # it if you keep a different number of channels. None makes
    # measure_crop skip pathogen measurements, so infection status
    # cannot be scored. Default 6.
    'pathogen_mask_dim': 6,

    # (list of lists) - Well locations of each pathogen condition, one
    # inner list per entry in pathogen_types. Every item must be a row
    # or column ID string such as 'c1' or 'r3'; anything else is
    # silently ignored and those wells stay unannotated. Ranges like
    # 'c2-c11' are not expanded - list each row/column. Do not leave it
    # None while pathogen_types is set: annotation is not skipped, every
    # row is labelled with the first pathogen_types entry. Defaults:
    # None in the plot-from-db settings,
    # [['c1','c2','c3'],['c4','c5','c6']] for recruitment analysis.
    'pathogen_plate_metadata': [['c1', 'c2', 'c3'], ['c4', 'c5', 'c6']],

    # (list) - Two-element [min, max] area filter in pixels squared
    # applied to the pathogen table in analyze_recruitment, well after
    # segmentation: rows with pathogen_area outside the open interval
    # are dropped. Bounds must be ints - floats are silently ignored.
    # None widens it to effectively unlimited. Default [0, 100000]. Use
    # it to discard debris and merged clumps.
    'pathogen_size_range': [0, 100000],

    # (list) - Names given to each pathogen condition on the plate, e.g.
    # ['wt','ku80']. Element i is written into the pathogen column for
    # every well listed in pathogen_plate_metadata[i] and folded into
    # the combined condition label used for grouping and plotting. Must
    # match pathogen_plate_metadata in length and order; None skips
    # pathogen annotation.
    'pathogen_types': ['pathogen_1', 'pathogen_2'],

    # (bool) - Render and save QC figures while the pipeline runs:
    # channel montages and Cellpose mask overlays during segmentation,
    # before/after filtration views and crop grids during measurement.
    # It adds figures per batch, so a full plate becomes much slower and
    # more memory-hungry; keep it for small or test_mode runs, which
    # force it on. Default False.
    'plot': True,

    # (bool) - Before the recruitment plots, draw a control panel of
    # per-compartment mean intensities (cell, nucleus, pathogen,
    # cytoplasm) for every channel, split by condition. Use it to
    # confirm channel assignment and that positive/negative control
    # wells separate as expected before trusting the recruitment
    # numbers. Turn it off to shorten the run. Default True.
    'plot_control': True,

    # (int) - How many merged image stacks from the start of the folder
    # are drawn with cell, nucleus and pathogen outlines overlaid before
    # recruitment analysis runs. The check is index <= plot_nr, so
    # plot_nr + 1 images actually appear and 0 still plots one. Raise it
    # to eyeball segmentation on more fields. Default 3.
    'plot_nr': 3,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (str) - Free-text label for the protein or marker imaged in
    # channel_of_interest, e.g. 'GRA1'. The recruitment run prints it in
    # its banner ('channel:3 = protein') to record what the recruitment
    # ratio is measuring; it feeds no computation, so changing it alters
    # nothing but that log line. Default 'protein'.
    'target': 'protein',

    # (float) - Recruitment-analysis cutoff on the 95th-percentile
    # intensity of channel_of_interest inside each cell: cells at or
    # below it are discarded before recruitment ratios are computed.
    # Raise it to keep only strongly expressing cells; set 0 or None to
    # disable the filter entirely. Raw intensity units, default 1.
    'target_intensity_min': 1,

    # (list of lists) - Plate wells that received each entry of
    # treatments, one inner list per treatment in the same order, e.g.
    # [['r1','r2','r3'],['r4','r5','r6']]. Entries must start with 'r'
    # (row) or 'c' (column); anything else is ignored and those wells
    # get no treatment label. Wells you do not list are still kept by
    # analyze_recruitment - 'condition' is the join of whatever
    # cell/pathogen/treatment labels exist and only rows missing all
    # three are dropped, and with the shipped cell_types default every
    # row has at least one label. plot_data_from_db is the path that
    # really filters: it drops rows with no treatment label whenever
    # this key is set. Default [['r1','r2','r3'],['r4','r5','r6']] in
    # the recruitment pipeline, None in the plot-from-db, endodyogeny
    # and class-proportion pipelines.
    'treatment_plate_metadata': [['r1', 'r2', 'r3'], ['r4', 'r5', 'r6']],

    # (list) - Names of the drug or treatment conditions in the
    # experiment, e.g. ['dmso','lovastatin']. Each name is written into
    # the treatment column and folded into the combined condition label
    # used for grouping and plotting; positionally paired with
    # treatment_plate_metadata (or treatment_loc), which lists the wells
    # for each. Default ['cm','lovastatin'].
    'treatments': ['cm', 'lovastatin'],

}

# Fill in anything left unset, then check the source path.
settings = get_analyze_recruitment_default_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
analyze_recruitment(settings)

## Where the output went

Per-object recruitment ratios and per-condition summary plots.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.